<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">وقتی گذشته واقعاً ثابت مانده است</h1>
<p style="text-align:right">درس 89 از 92 · کدام محاسبهٔ تولید را می‌توان دوباره استفاده کرد؟ · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">64-cache</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-10/chapter-01/64-cache.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right">خروجی آخرین <bdi dir="ltr">Query</bdi> را با <bdi dir="ltr">K/V</bdi> نگه‌داشته‌شده بازسازی کنید و مرز اعتبار <bdi dir="ltr">Cache</bdi> را ببینید.</p><p style="text-align:right">پیش‌نیاز: <bdi dir="ltr">Q/K/V</bdi> با شکل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,H,T,D)</code>، حالت <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">eval</code> و موقعیت‌های ثابت.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۴۵–۸۰ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر پنجره از چپ بریده و موقعیت‌ها از صفر شماره‌گذاری شوند، آیا <bdi dir="ltr">K/V</bdi> قبلی هنوز همان محاسبه را نشان می‌دهند؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
torch.set_num_threads(1)
torch.manual_seed(7)
model = MiniGPT(ModelConfig(12,4,8,2,1,0.0)).eval()
prefix = torch.tensor([[1,2,3]])
extended = torch.tensor([[1,2,3,4]])
with torch.no_grad():
    old_trace,new_trace = {},{}
    model(prefix,trace=old_trace)
    model(extended,trace=new_trace)
old_attention = old_trace['layers'][0]['attention']
new_attention = new_trace['layers'][0]['attention']
torch.testing.assert_close(old_attention['k'],new_attention['k'][:,:,:3])
print('old/new key shapes:',tuple(old_attention['k'].shape),tuple(new_attention['k'].shape))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">cached_last_attention(q_new,k_old,v_old,k_new,v_new)</code> سه <bdi dir="ltr">Tensor</bdi> برگرداند: خروجی <bdi dir="ltr">Query</bdi> تازه، <bdi dir="ltr">K</bdi> کامل و <bdi dir="ltr">V</bdi> کامل. همه شکل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,H,T,D)</code> دارند و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">q_new/k_new/v_new</code> فقط یک موقعیت دارند؛ الحاق روی محور زمان و مقیاس <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">1/sqrt(D)</code> است. آخرین <bdi dir="ltr">Query</bdi> اجازهٔ دیدن همهٔ این موقعیت‌ها را دارد.</p>
</div>

In [ ]:
def cached_last_attention(q_new, k_old, v_old, k_new, v_new):
    # TODO: فقط محاسبهٔ آخرین Query
    return None

In [ ]:
def test_exercise():
    args = (new_attention['q'][:,:,-1:],old_attention['k'],old_attention['v'],
            new_attention['k'][:,:,-1:],new_attention['v'][:,:,-1:])
    result = cached_last_attention(*args)
    if result is None:
        return False
    output,keys,values = result
    torch.testing.assert_close(keys,new_attention['k'])
    torch.testing.assert_close(values,new_attention['v'])
    torch.testing.assert_close(output,new_attention['weighted_values'][:,:,-1:])
    query = torch.ones(1,1,1,2)
    empty = torch.empty(1,1,0,2)
    value = torch.tensor([[[[3.0,5.0]]]])
    one = cached_last_attention(query,empty,empty,query,value)
    torch.testing.assert_close(one[0],value)
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: cached_last_attention')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط طول <bdi dir="ltr">Prefix</bdi> را از ۱ تا ۳ افزایش دهید؛ <bdi dir="ltr">K/V</bdi> ذخیره‌شده را بشمارید. این شمارش، اندازهٔ <bdi dir="ltr">Cache</bdi> است نه حافظهٔ کل مدل.</p>
</div>

In [ ]:
with torch.no_grad():
    for length in (1,2,3):
        trace = {}
        model(extended[:,:length],trace=trace)
        attention = trace['layers'][0]['attention']
        print(length,'K+V elements:',attention['k'].numel()+attention['v'].numel())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">حذف قدیمی‌ترین <bdi dir="ltr">K/V</bdi> کافی نیست، چون شمارهٔ موقعیت بقیه هم عوض شده است. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">cache_reusable(old_ids,new_ids,limit)</code> برای فهرست‌های <bdi dir="ltr">ID</bdi> و فرض وزن و حالت ثابت، فقط وقتی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">True</code> بدهد که دقیقاً یک <bdi dir="ltr">Token</bdi> به <bdi dir="ltr">Prefix</bdi> بدون تغییر اضافه شده و طول از <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">limit</code> نگذشته باشد.</p>
</div>

In [ ]:
with torch.no_grad():
    shifted = {}
    model(torch.tensor([[2,3,4,5]]),trace=shifted)
stale_keys = new_attention['k'][:,:,1:]
recomputed_keys = shifted['layers'][0]['attention']['k'][:,:,:3]
print('stale/recomputed key difference:',(stale_keys-recomputed_keys).abs().max().item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def cache_reusable(old_ids, new_ids, limit):
    # TODO: اعتبار Prefix و موقعیت‌ها، پیش از استفادهٔ دوباره
    return None

In [ ]:
def test_repair():
    result = cache_reusable([1,2],[1,2,3],4)
    if result is None:
        return False
    assert result is True
    assert cache_reusable([1,2],[1,4,3],4) is False
    assert cache_reusable([1,2,3,4],[1,2,3,4,5],4) is False
    assert cache_reusable([1,2],[1,2],4) is False
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: cache_reusable')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right"><bdi dir="ltr">Q/K/V</bdi> از <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">trace</code> واقعی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">MiniGPT</code> آمده‌اند؛ تابع شما فقط هستهٔ <bdi dir="ltr">Attention</bdi> آخرین موقعیت را بازسازی می‌کند، نه <bdi dir="ltr">Cache</bdi> کامل همهٔ <bdi dir="ltr">Layer</bdi>‌ها. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">generate</code> پروژه همچنان کل پنجره را دوباره محاسبه می‌کند.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">برای تبدیل این تمرین به <bdi dir="ltr">Cache</bdi> کامل مدل، کدام وضعیت‌ها را باید برای هر <bdi dir="ltr">Layer</bdi> نگه دارید؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-10/chapter-01/64-cache.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/64-cache.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>